# Qdrant Vector Database - Practical Guide

## 1. Running Qdrant via Docker

### Pull Image
```bash
docker pull qdrant/qdrant:latest
```

### Run Container
```bash
docker run -d --name qdrant-learn -p 6333:6333 -p 6334:6334 qdrant/qdrant:latest
```

### Dashboard UI
Access the local web dashboard at: [http://localhost:6333/dashboard](http://localhost:6333/dashboard)


## 2. Client Initialization

- `QdrantClient`: Core client interface for REST and gRPC operations.
- `models`: Schema definitions (`VectorParams`, `Distance`, `PointStruct`).


In [1]:
from qdrant_client import QdrantClient, models

# Connect to local Qdrant container
client = QdrantClient(url="http://localhost:6333")

# In-memory instance for testing:
# client = QdrantClient(":memory:")

print("Connected to Qdrant server.")


Connected to Qdrant server.


## 3. Creating Collections

A collection groups points with a unified vector schema:
- `size`: Embedding vector dimension (e.g., 1536 for OpenAI, 384 for MiniLM/FastEmbed, 4 for testing).
- `distance`: Distance metric for similarity ranking (`COSINE`, `DOT`, `EUCLID`).


In [2]:
collection_name = "first_collection"

if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=4,
            distance=models.Distance.COSINE,
        )
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")


Collection 'first_collection' already exists.


### Inspecting Collections


In [ ]:
collections = client.get_collections()  # CollectionsResponse( collections=[]) # object inside it [ list of collections ]
print("Available collections:", [col.name for col in collections.collections])  


Available collections: ['chunks_fixed', 'chunks_recursive', 'documents', 'first_collection', 'langchain_demo', 'movies_collection']


## 4. Points (Vectors + Payload)

A **Point** is the atomic data unit stored in a collection:
1. `id`: Unique identifier (64-bit integer or UUID).
2. `vector`: Float array matching the collection dimension.
3. `payload`: Metadata dictionary (JSON) containing document details.


In [4]:
points = [
    models.PointStruct( # points 
        id=1,
        vector=[0.1, 0.2, 0.3, 0.4],
        payload={"category": "example"} # fields
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.3, 0.4, 0.5],
        payload={"category": "demo"}
    ),
]

client.upsert(
    collection_name=collection_name,
    points=points
)
print(f"Upserted {len(points)} points successfully.")


Upserted 2 points successfully.


### Collection Statistics


In [ ]:
collection_info = client.get_collection(collection_name) # CollectionInfo object
print(f"Status: {collection_info.status} | Points count: {collection_info.points_count} | indexed_vector {collection_info.indexed_vectors_count} ") 


Status: green | Points count: 2 | 0


## 5. Vector Similarity Search

Querying the vector index with a search vector to retrieve nearest neighbors by cosine similarity.


In [8]:
query_vector = [0.05, 0.60, 0.75, 0.70]

# Returns QueryResponse containing ScoredPoint records
result = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=2
) # return  -> points object

print(result) 

for point in result.points:
    print(f"ID: {point.id} | Score: {point.score:.4f} | Payload: {point.payload}")


points=[ScoredPoint(id=1, version=2, score=0.9669448, payload={'category': 'example'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2, version=2, score=0.9609574, payload={'category': 'demo'}, vector=None, shard_key=None, order_value=None)]
ID: 1 | Score: 0.9669 | Payload: {'category': 'example'}
ID: 2 | Score: 0.9610 | Payload: {'category': 'demo'}


## 6. Metadata Filtering (Payload Filter)

Combining vector similarity with exact payload conditions.


In [ ]:
search_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="category",
            match=models.MatchValue(value="demo")
        )
    ]
)

filter_result = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=search_filter,
    limit=1
)

for point in filter_result.points:
    print(f"Filtered Result -> ID: {point.id} | Score: {point.score:.4f} | Payload: {point.payload}")


## 7. Point ID Formats

Qdrant enforces strict ID formats for points:
1. **64-bit Unsigned Integers** (`1, 2, 3, ...`)
2. **UUID strings** (`str(uuid.uuid4())`) -> Universally Unique Identifier

Arbitrary strings (e.g. `"chunk_0"`) will raise validation errors.


## 8. Vector Representations (Dense, Sparse, Multi-Vector)

### 1. Dense Vectors
- **Structure:** Fixed-length array of continuous floats (e.g., 384 or 1536 dimensions).
- **Focus:** High-level semantic meaning, conceptual similarity, cross-lingual context.
- **Models:** OpenAI `text-embedding-3`, BAAI `bge-m3`, HuggingFace `all-MiniLM-L6-v2`.
- **Trade-off:** Strong for paraphrasing; weaker for exact serial numbers, SKUs, or code terms.

---

### 2. Sparse Vectors
- **Structure:** High-dimensional array (vocabulary size, 30,000+ dimensions) where most values are zero.
- **Focus:** Exact keyword matching and term frequency weighting.
- **Models / Algorithms:** BM25, TF-IDF, SPLADE.
- **Trade-off:** High precision for rare tokens and exact keywords; no semantic abstraction.

---

### 3. Multi-Vectors (Late Interaction)
- **Structure:** A matrix of vectors per document (one vector per token).
- **Focus:** Preserving token-level granularity without bottleneck compression into a single vector.
- **Models:** ColBERT (text), ColPali (document vision / PDFs).
- **Trade-off:** Maximum retrieval precision; requires higher memory and storage.

---

### Comparison Matrix

| Feature | Dense Vectors | Sparse Vectors | Multi-Vectors |
| :--- | :--- | :--- | :--- |
| **Output Shape** | Single fixed vector | Single sparse vector | Array of token vectors |
| **Primary Target** | Semantic context | Exact keywords & terms | Token-level nuance |
| **Common Models** | MiniLM, OpenAI, BGE | BM25, SPLADE | ColBERT, ColPali |
| **Use Case** | General Q&A, conceptual RAG | Technical codes, IDs, names | Complex multi-page documents |

---

### Hybrid Search: Anatomy of Collection Configuration

When configuring a collection with multiple vector types (Hybrid Search), observe the distinction between **fixed API parameters** and **custom vector aliases**:

| Component | Nature | Purpose | Example |
| :--- | :--- | :--- | :--- |
| **`vectors_config`** | Fixed API Parameter | Reserved by `create_collection` for dense vector definitions. | Cannot be renamed. |
| **`sparse_vectors_config`** | Fixed API Parameter | Reserved by `create_collection` for sparse vector definitions. | Cannot be renamed. |
| **`"text_dense"`** | Custom Vector Alias | User-defined identifier for the dense vector. | Can be named `"semantic"`, `"dense"`, etc. |
| **`"text_sparse"`** | Custom Vector Alias | User-defined identifier for the sparse vector. | Can be named `"bm25"`, `"keywords"`, etc. |

#### Why `SparseVectorParams()` has no `size` parameter:
- **Dense:** Requires fixed capacity allocation in memory (`size=1536`, `distance=COSINE`).
- **Sparse:** Dimensions correspond dynamically to token IDs in vocabulary; similarity defaults to dot product.


In [ ]:
# Hybrid Collection Configuration (Named Dense + Named Sparse)
client.create_collection(
    collection_name="hybrid_collection",
    
    # 1. Dense vectors configuration (Fixed parameter name)
    vectors_config={
        "text_dense": models.VectorParams(  # Custom alias for the dense vector
            size=1536,
            distance=models.Distance.COSINE
        )
    },
    
    # 2. Sparse vectors configuration (Fixed parameter name)
    sparse_vectors_config={
        "text_sparse": models.SparseVectorParams()  # Custom alias; size is dynamic, distance defaults to dot product
    }
)

print("Hybrid collection configured successfully.")


### Embedding Model Types
1. **Integrated Frameworks:** `FastEmbed` (native Qdrant integration, ONNX-based execution).
2. **Open-Source Local Models:** `sentence-transformers` (HuggingFace, PyTorch/Transformers).
3. **Managed Cloud APIs:** OpenAI, Google Gemini, Cohere.


## 9. Payload Architecture (Metadata)

Payload stores structured document metadata alongside the vector in JSON format, avoiding external database lookups during retrieval.

### Supported Types
- **Textual:** Document text, category, author, tags, source file.
- **Numeric:** Timestamps, prices, ratings, token counts, page numbers.
- **Booleans & Arrays:** Flags (`is_active`), multi-tag arrays (`["ai", "python"]`).

### Operational Roles
1. **Pre-Filtering:** Restricting search space by metadata before or during vector ranking.
2. **In-Place Retrieval:** Returning original text and source references directly from the vector index.


## 10. Document Chunking for Vector Search

### Technical Rationale
1. **Model Capacity:** Embedding models enforce fixed token limits (e.g., 256 tokens for `all-MiniLM-L6-v2`). Exceeding this causes silent truncation.
2. **Information Density:** Compressing large documents into a single vector washes out specific details.
3. **Retrieval Precision:** Search should point to the exact relevant passage rather than an entire document.

```text
Raw Document ──▶ Text Splitter ──▶ Chunks ──▶ Embedding Model ──▶ Qdrant PointStruct
```

### Strategies
- **Fixed-Size Chunking:** Character-based slicing. Fast, but splits across word and sentence boundaries.
- **Recursive Character Chunking:** Slices along natural structural separators (`\n\n`, `\n`, spaces).
- **Token-Aware Chunking:** Uses the model tokenizer directly to guarantee chunks fit within capacity.

### Sizing Principles
- **Chunk Size:** 200 to 500 tokens for general text.
- **Chunk Overlap:** 10% to 20% to prevent context loss at chunk boundaries.


In [ ]:
# Chunk Storage with Structured Payload
doc_chunks = [
    "Artificial intelligence is a branch of computer science focused on building learning systems.",
    "Vector databases like Qdrant index and retrieve high-dimensional semantic representations.",
    "Text chunking breaks long documents into coherent passages to maintain search precision."
]

chunks_collection = "demo_chunks_collection"
if not client.collection_exists(chunks_collection):
    client.create_collection(
        collection_name=chunks_collection,
        vectors_config=models.VectorParams(
            size=4,
            distance=models.Distance.COSINE
        )
    )

chunk_points = [
    models.PointStruct(
        id=i + 1,
        vector=[0.1 * (i + 1), 0.2, 0.3, 0.4],
        payload={
            "text": chunk,
            "chunk_id": i + 1,
            "doc_id": "ai_overview_doc",
            "char_count": len(chunk)
        }
    )
    for i, chunk in enumerate(doc_chunks)
]

client.upsert(
    collection_name=chunks_collection,
    points=chunk_points
)

print(f"Upserted {len(chunk_points)} chunks to '{chunks_collection}'.")


## Sparse Vector Representation

Sparse vectors have high dimensionality (vocabulary size, 30,000+ dimensions), but most values are zeros.
Instead of storing the entire vector, we store only the non-zero positions and their weights:

```text
Full Vector:   [0.0, 0.7, 0.0, 0.9, 0.0, ...]
               
Stored As:
  indices = [1, 3]        (positions of non-zero values)
  values  = [0.7, 0.9]    (corresponding weights)
```


### Inverted Index

An inverted index organizes data by **dimension index** rather than by chunk, enabling direct lookups instead of full scans:

```text
Forward Index (by Chunk):            Inverted Index (by Dimension):
  Chunk 1 -> [Dim 0: 0.8, Dim 1: 0.6]     Index 0 -> Chunk 1: 0.8, Chunk 2: 0.5
  Chunk 2 -> [Dim 0: 0.5]                  Index 1 -> Chunk 1: 0.6, Chunk 3: 0.4
  Chunk 3 -> [Dim 1: 0.4]
```

When a query hits Index 0, only Chunk 1 and Chunk 2 are scored. Chunk 3 is skipped entirely.


In [14]:
# Create a collection with sparse vector configuration
collection_name = "sparse_collection"


if client.collection_exists(collection_name):
    client.delete_collection(collection_name)


client.create_collection(
    collection_name=collection_name,
    sparse_vectors_config={
        "text_sparse": models.SparseVectorParams(
            index=models.SparseIndexParams(
                full_scan_threshold=0,            # 0 = always use inverted index; N = points threshold before switching from brute-force
                on_disk=False,                    # False = store index in RAM for low latency
                datatype=models.Datatype.FLOAT32  # Storage precision for sparse values
            )
        )
    }
)

print(f"Sparse collection '{collection_name}' created successfully.")

Sparse collection 'sparse_collection' created successfully.


In [16]:
# Insert points with sparse vectors
client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=1,
            payload={"text": "sample document"},
            vector={
                "text_sparse": models.SparseVector(
                    indices=[0, 1],      # uint32: positions of non-zero tokens/features
                    values=[0.1, 0.1]    # float32: weights of corresponding indices
                )
            }
        ),
        models.PointStruct(
            id=1,
            payload={"text": "sample document"},
            vector={
                "text_sparse": models.SparseVector(
                    indices=[1, 2, 3],      # uint32: positions of non-zero tokens/features
                    values=[0.2, -0.2 , 0.2]    # float32: weights of corresponding indices
                )
            }
        )
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
# Query sparse collection using the target named vector
response = client.query_points(
    collection_name=collection_name,
    using="text_sparse",                 # Target sparse vector name
    limit=1,
    query=models.SparseVector(
        indices=[1, 3],                  # Query non-zero token positions
        values=[1.0, 1.0]                # Query weights
    ),
    with_vectors=True                    # Include vectors in search response
)

response

QueryResponse(points=[ScoredPoint(id=1, version=1, score=0.4, payload={'text': 'sample document'}, vector={'text_sparse': SparseVector(indices=[1, 2, 3], values=[0.2, -0.2, 0.2])}, shard_key=None, order_value=None)])

## BM25 (Best Matching 25) Implementation

Dependency installation:
```bash
uv add fastembed
```

---

### How BM25 & Inverted Index Work

#### Example Setup:
* **D1:** `"Python Python AI"`
* **D2:** `"Python ML"`
* **D3:** `"AI ML ML"`
* **Query:** `"Python AI"`

#### 1. Tokenization
Split query into individual tokens:
* `Tokens = ["Python", "AI"]`

#### 2. Inverted Index Lookup
Direct lookup of documents containing query tokens:
* `"Python"` -> **[D1, D2]**
* `"AI"`     -> **[D1, D3]**

#### 3. BM25 Scoring Components
For each matching token, BM25 computes weights using:
* **TF (Term Frequency):** How often the word appears in the document.
* **IDF (Inverse Document Frequency):** How rare the word is across all documents.
* **Length Normalization (`avg_len`):** Normalizes scores against average document length.

#### 4. Score Aggregation & Ranking
Sum token scores per document:
* **Score(D1):** Score(`"Python"`) + Score(`"AI"`) *(matched both)*
* **Score(D2):** Score(`"Python"`)
* **Score(D3):** Score(`"AI"`)

**Final Ranking:** `D1 > D2, D3`

> **Note:** BM25 is lexical (exact keyword matching with statistical weights), unlike Dense embeddings which capture semantic meaning and synonyms.


In [ ]:
from qdrant_client import QdrantClient,models
client = QdrantClient(url="http://localhost:6333")

In [ ]:
collection_name = "my_sparse_collection"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config={},  # Required empty dict when using only sparse vectors (no dense vectors)
    sparse_vectors_config={
        "bm25_sparse_vector": models.SparseVectorParams(
            modifier=models.Modifier.IDF  # Dynamically calculate collection-wide IDF weights for BM25
        )
    }
)

print(f"Collection '{collection_name}' created successfully.")

In [ ]:
grocery_items_descriptions = [
    "Grated hard cheese",
    "White crusty bread roll",
    "Mac and cheese"
]

# Calculate corpus average document length for BM25 length normalization
avg_document_length = sum(len(description.split()) for description in grocery_items_descriptions) / len(grocery_items_descriptions)
print(f"Average document length: {avg_document_length}")

client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=i,
            payload={"text": description},
            vector={
                "bm25_sparse_vector": models.Document(
                    text=description,
                    model="Qdrant/bm25",  # Automatic sparse tokenization via FastEmbed
                    options={"avg_len": avg_document_length}  # BM25 document length calibration
                )
            },
        ) for i, description in enumerate(grocery_items_descriptions)
    ]
)

average document length: 3.3333333333333335


In [ ]:
# Query points using BM25 sparse vector
response = client.query_points(
    collection_name=collection_name,
    using="bm25_sparse_vector",  # Target sparse vector index
    query=models.Document(
        text="cheese",
        model="Qdrant/bm25"
    ),
    with_vectors=True  # Include indices and weights in response
)

response

## SPLADE (Sparse Lexical and Expansion Model)

SPLADE is a **Neural Sparse** retrieval model (based on BERT) that solves the vocabulary mismatch problem:
* **Term Expansion:** Automatically predicts related terms (e.g., query `"not soft"` expands to activate the token `"hard"`).
* **Neural Weighting:** Assigns learned importance scores to tokens instead of simple frequency counts.
* **No IDF Modifier Needed:** SPLADE vectors are already weighted by the neural network, so we don't apply `Modifier.IDF`.

In [ ]:
collection_name = "splade_sparse_collection"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config={},  # No dense vectors
    sparse_vectors_config={
        # SPLADE vectors contain learned weights; no collection-wide modifier needed
        "splade_sparse_vector": models.SparseVectorParams()
    }
)

print(f"Collection '{collection_name}' created successfully.")

In [ ]:
# Upsert documents using SPLADE for neural sparse embeddings
client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=i,
            payload={"text": description},
            vector={
                "splade_sparse_vector": models.Document(
                    text=description,
                    model="prithivida/Splade_PP_en_v1"  # FastEmbed downloads and runs ONNX weights
                )
            }
        )
        for i, description in enumerate(grocery_items_descriptions)
    ]
)

print(f"Upserted {len(grocery_items_descriptions)} documents into '{collection_name}'.")

In [ ]:
# 1. Query with SPLADE (Neural Sparse)
# SPLADE expands "A not soft cheese" -> matches "Grated hard cheese" via semantic term expansion
response_splade = client.query_points(
    collection_name="splade_sparse_collection",
    using="splade_sparse_vector",
    query=models.Document(
        text="A not soft cheese",
        model="prithivida/Splade_PP_en_v1"
    ),
    limit=3
)

print("=== SPLADE Search Results ===")
for point in response_splade.points:
    print(f"Score: {point.score:.4f} | Text: {point.payload['text']}")

In [ ]:
# 2. Query with BM25 (Exact Keyword Match Comparison)
# BM25 searches for exact words ("not", "soft", "cheese") and misses the connection to "hard"
response_bm25 = client.query_points(
    collection_name="my_sparse_collection",  # Query the BM25 collection
    using="bm25_sparse_vector",
    query=models.Document(
        text="A not soft cheese",
        model="Qdrant/bm25"
    ),
    limit=3
)

print("=== BM25 Search Results ===")
for point in response_bm25.points:
    print(f"Score: {point.score:.4f} | Text: {point.payload['text']}")

## Hybrid Search (Dense + Sparse)

Combines two complementary retrieval mechanisms:
* **Dense Vectors (`all-MiniLM-L6-v2`):** Semantic meaning, context, and conceptual similarity.
* **Sparse Vectors (`BM25`):** Exact lexical keyword matching and specific entity lookup.

In [ ]:
collection_name = "hybrid_search_demo"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            size=384,                             # Dimension of all-MiniLM-L6-v2
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF          # Dynamic collection-wide IDF for BM25
        )
    }
)

print(f"Hybrid collection '{collection_name}' created successfully.")

In [ ]:
# Sample cheese and culinary corpus for hybrid retrieval demonstration
documents = [
    "Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation.",
    "Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging.",
    "Brie cheese features a soft, creamy interior surrounded by an edible white rind.",
    "This French cheese has a flowing, buttery center encased in a bloomy white crust.",
    "Fresh mozzarella pairs beautifully with ripe tomatoes and basil leaves.",
    "Classic Margherita pizza topped with tomato sauce, mozzarella, and fresh basil.",
    "Parmesan requires at least 12 months of cave aging to develop its signature sharp taste.",
    "Parmigiano-Reggiano's distinctive piquant flavor comes from extended maturation in controlled environments.",
    "Grilled cheese sandwiches are the ultimate American comfort food for cold winter days.",
    "Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich.",
]

In [ ]:
import uuid

# Upsert documents with simultaneous dense and sparse vector generation via FastEmbed
client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=str(uuid.uuid4()),
            payload={"text": doc},
            vector={
                "dense": models.Document(
                    text=doc,
                    model="sentence-transformers/all-MiniLM-L6-v2"
                ),
                "sparse": models.Document(
                    text=doc,
                    model="Qdrant/bm25"
                )
            }
        )
        for doc in documents
    ]
)

print(f"Successfully upserted {len(documents)} hybrid documents.")

### Reciprocal Rank Fusion (RRF)

RRF combines search results from multiple retrievers based on **rank position** rather than raw scores:
* **Why Rank instead of Score?** Dense scores (Cosine: $0$ to $1$) and Sparse scores (BM25: $0$ to $\infty$) operate on incompatible scales and cannot be directly summed without distortion.
* **How RRF Works:** Documents that appear near the top in both lists receive the highest fused score.

#### Formula:
$$RRF(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

* $d$: The candidate document.
* $r \in R$: Ranked list from each retriever (Dense list, Sparse list).
* $\text{rank}_r(d)$: 1-based rank position of document $d$ in retriever $r$.
* $k$: Smoothing constant (standard default is **60**) that prevents top-ranked items from dominating disproportionately.

In [ ]:
query_text = "aged cheese with nutty crystalline taste"

# Execute hybrid search fusing Dense and Sparse rankings via RRF
response = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(
            query=models.Document(
                text=query_text,
                model="sentence-transformers/all-MiniLM-L6-v2"
            ),
            using="dense",
            limit=5
        ),
        models.Prefetch(
            query=models.Document(
                text=query_text,
                model="Qdrant/bm25"
            ),
            using="sparse",
            limit=5
        )
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3
)

print(f"Query: '{query_text}'\n")
print("=== RRF Hybrid Search Results ===")
for point in response.points:
    print(f"Score: {point.score:.5f} | Text: {point.payload['text']}")

In [ ]:
def RRF_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF), 
        limit=3,
    )
    return response.points  


In [ ]:
queries = [
    "nutty aged cheese",
    "soft French cheese",
    "pizza ingredients",
    "a good lunch",
]
for query in queries:
    print("Query:", query)
    
    rff_results= RRF_search(query)
    print("RRF Results:")
    for result in rff_results:
        print("\t-",result.payload['text'],result.score)
    print()    

### Distribution-Based Score Fusion (DBSF)

Unlike RRF (which only looks at rank positions and discards raw scores), **DBSF** preserves relative score margins by statistically normalizing the distributions:
* **How it Works:** It estimates the mean ($\mu$) and standard deviation ($\sigma$) of scores returned by each retriever, scales them to a unified distribution, and computes a fused score.
* **Key Advantage:** If one document is a massive standout match in Dense search (far ahead of other candidates), DBSF retains that strong signal instead of treating it as just "rank 1".
* **RRF vs DBSF Rule of Thumb:** 
  * Use **RRF** as a safe, robust default across disparate score metrics.
  * Use **DBSF** when score margins convey critical confidence information.

In [ ]:
def dbsf_search(query: str, limit: int = 3) -> list[models.ScoredPoint]:
    """Execute hybrid search fusing Dense and Sparse retrievers using DBSF."""
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=10,  # Wider candidate pool for accurate mean/std calculation
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=10,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.DBSF),
        limit=limit,
    )
    return response.points

In [ ]:
# Evaluate test queries using DBSF score fusion
for query in queries:
    print(f"Query: '{query}'")
    dbsf_results = dbsf_search(query, limit=3)
    print("DBSF Results:")
    for result in dbsf_results:
        print(f"  - [{result.score:.4f}] {result.payload['text']}")
    print("-" * 60)